In [1]:
from datasets import Dataset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re


c:\Users\agsto\Desktop\CMPE-252-summary-project\summary_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer,DataCollatorForSeq2Seq, Seq2SeqTrainingArguments
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())       # True if GPU is usable
print("CUDA version:", torch.version.cuda)                # CUDA version PyTorch was built with
print("Number of GPUs:", torch.cuda.device_count())       # Number of GPUs detected
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4070 SUPER


## Importing and cleaning data set

In [3]:
section_path = "section_train_0.parquet" #file path
section = pd.read_parquet(section_path)

In [4]:
documention_path_0 = "document_train_0.parquet" #file path
document_0 = pd.read_parquet(documention_path_0)

documention_path_1 = "document_train_1.parquet"
document_1 = pd.read_parquet(documention_path_1)

documention_path_2 = "document_train_2.parquet"
document_2 = pd.read_parquet(documention_path_2)

documention_path_3 = "document_train_3.parquet"
document_3 = pd.read_parquet(documention_path_3)

In [5]:
document = pd.concat([document_0, document_1, document_2, document_3])

In [6]:
document

,article,abstract
0,additive models @xcite provide an important fa...,additive models play an important role in semi...
1,the leptonic decays of a charged pseudoscalar ...,"we have studied the leptonic decay @xmath0 , v..."
2,the transport properties of nonlinear non - eq...,"in 84 , 258 ( 2000 ) , mateos conjectured that..."
3,studies of laser beams propagating through tur...,the effect of a random phase diffuser on fluct...
4,the so - called `` nucleon spin crisis '' rais...,with a special intention of clarifying the und...
...,...,...
13531,there has been a recent upsurge in interest in...,we present measurements of the integrated flux...
13532,the last few years have seen growing evidence ...,we show how a very accurate measurement of the...
13533,"the observed initial mass function of stars , ...",we demonstrate the feasibility of detecting di...
13534,numerical simulations of the growth of large s...,a clustering analysis is performed on two samp...


In [14]:
def clean_document(document):
    document['abstract'] = document['abstract'].str.replace(' \n', '', regex=False) #remove \n
    document['abstract'] = document['abstract'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['abstract'] = document['abstract'].str.replace('  ', ' ', regex=False) #replace double space with single space

    document['article'] = document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['article'] = document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

    article_summary = document['article'].str.len().describe()
    abstract_summary = document['abstract'].str.len().describe()

    document = document[document['article'].str.len() >= article_summary['25%']] 
    document = document[document['article'].str.len() <= article_summary['75%']]  

    document = document[document['abstract'].str.len() >= abstract_summary['25%']]
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document[document['article'].str.len() >= article_summary['25%']] 
    document = document[document['article'].str.len() <= article_summary['75%']]  

    document = document[document['abstract'].str.len() >= abstract_summary['25%']]
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document.drop_duplicates()
    document = document.reset_index(drop=True)
    
    return document

In [19]:
document = clean_document(document)

## BART (No training yet)

In [10]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, output_attentions=True)

model.config.output_attentions = True
model.config.return_dict = True

model.eval()

Please make sure the generation config includes `forced_bos_token_id=0`. 
The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1105.47it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

## Validation and Test cleaning

In [16]:
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, output_attentions=True, torch_dtype = torch.bfloat16)

Loading weights: 100%|██████████| 511/511 [00:00<00:00, 726.66it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


In [10]:
validation_path = "document_validation.parquet" #file path
validation = pd.read_parquet(validation_path)

test_path = "document_test.parquet"
test = pd.read_parquet(test_path)

In [16]:
validation = clean_document(validation)

In [17]:
test = clean_document(test)

## Training

In [24]:
train_dataset = Dataset.from_pandas(document)
validation_dataset   = Dataset.from_pandas(validation)
test_dataset  = Dataset.from_pandas(test)

In [ ]:
label = tokenizer(
    list(train_dataset["abstract"]),
    truncation=True,
    max_length=1024,
    padding=False
)

# Get lengths
label_lengths = [len(ids) for ids in label["input_ids"]]

In [31]:
abstract_tokens_summary = pd.DataFrame(label_lengths).describe()[0]
abstract_tokens_summary

count    15490.000000
mean       193.410781
std         42.827134
min        106.000000
25%        159.000000
50%        189.000000
75%        223.000000
max        544.000000
Name: 0, dtype: float64

In [32]:
token_len_IQR =  abstract_tokens_summary['75%'] - abstract_tokens_summary['25%']
1.5 * token_len_IQR

np.float64(96.0)

In [33]:
abstract_tokens_summary['25%'] - (1.5 * token_len_IQR)

np.float64(63.0)

In [34]:
abstract_tokens_summary['75%'] + (1.5 * token_len_IQR)

#pad to 320 should be a clean number

np.float64(319.0)

In [32]:
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746").cuda()
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1132.74it/s, Materializing param=model.shared.weight]                                   


In [28]:
def tokenize_document(dataset):
    model_input = tokenizer(dataset["article"], max_length = 1024, truncation = True)
    labels = tokenizer(dataset["abstract"], max_length = 320, truncation = True)

    model_input["labels"] = labels['input_ids']
    return model_input

In [30]:
tokenized_dataset = train_dataset.map(tokenize_document, batched=True)
validation_tokenized = validation_dataset.map(tokenize_document, batched=True, remove_columns=validation_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_document, batched=True, remove_columns=test_dataset.column_names)

Map: 100%|██████████| 1715/1715 [00:05<00:00, 298.92 examples/s]


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest"  # pads to the longest sequence in each batch
)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetuned",      # folder to save checkpoints
    num_train_epochs=3,                  # number of passes through the dataset
    per_device_train_batch_size=4,       # safe for 12GB VRAM
    per_device_eval_batch_size=4,        # same as train batch size
    learning_rate=2e-5,                  # learning rate
    weight_decay=0.01,                   # regularization
    save_total_limit=3,                   # keep last 3 checkpoints
    eval_strategy="epoch",          # evaluate once per epoch
    save_strategy="epoch",                # save checkpoint once per epoch
    logging_strategy="steps",             # log every N steps
    logging_steps=100,                    # adjust if needed
    bf16=True,                            # mixed precision for VRAM efficiency
    predict_with_generate=True            # necessary for seq2seq evaluation
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=validation_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train(resume_from_checkpoint="./bart_finetuned/checkpoint-7746") #pick up from Epoch = 2

### Epoch	Training Loss	Validation Loss
### 1	2.795941	2.686332
### 2	2.850331	2.683255
### 3	2.777663	2.682559

## ROUGE

In [58]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

In [59]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # If model returns tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=True
    )

    # Replace -100 in labels (ignore index) so they can be decoded
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True
    )

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"]
    }

## Trained model ROUGE

In [60]:
#epoch = 3

rouge = evaluate.load("rouge")

predictions = trainer.predict(validation_tokenized)

preds = predictions.predictions
labels = predictions.label_ids

labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

results = rouge.compute(
    predictions=decoded_preds,
    references=decoded_labels,
    use_stemmer=True
)

print({k: round(v * 100, 4) for k, v in results.items()})

{'rouge1': np.float64(44.6801), 'rouge2': np.float64(15.9898), 'rougeL': np.float64(24.713), 'rougeLsum': np.float64(24.7124)}


### ROUGE results from epoch = 2

{'rouge1': np.float64(44.6586), 'rouge2': np.float64(15.9589), 'rougeL': np.float64(24.7595), 'rougeLsum': np.float64(24.7636)}

### ROUGE results from epoch = 3

{'rouge1': np.float64(44.6801), 'rouge2': np.float64(15.9898), 'rougeL': np.float64(24.713), 'rougeLsum': np.float64(24.7124)}

# Summary Generation comparison

In [22]:
from transformers import BartForConditionalGeneration, BartTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746").to(device)
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

inputs = tokenizer(
    test['article'][1],
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=1024   # BART max input length
).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=320,
        min_length=30,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1223.86it/s, Materializing param=model.shared.weight]                                   


we study in details the problem of sensitivity loss due to discretization of parameters and to the needs to limit the computing cost , with hough procedures . we propose and study the characteristics of a frequency hough procedure , designed mainly to reduce the discretized problem , and we compare it with the sky hough method , which is actually used in the virgo collaboration . we present the study of amplitude losses due to digitization , and thus efficiencies , for both the procedures , and compare them with the rome hierarchical procedure , based on the main idea of coincidences among subsets of data . we show that the new frequency procedure has a better efficiency when the data sets have similar sensitivities , which are mainly due to the complexity of the digitization of the parameters . we also discuss the effects of digitization on the sensitivity of parameters , and show that for the new procedure , the sensitivity loss is due mainly to the effect of this complexity , which 

In [21]:
test['abstract'][1]

'in the hierarchical search for periodic sources of gravitational waves , the candidate selection , in the incoherent step , can be performed with hough transform procedures . in this paper we analyze the problem of sensitivity loss due to discretization of the parameters space vs computing cost , comparing the properties of the sky hough procedure with those of a new frequency hough , which is based on a transformation from the _ time - observed frequency _ plane to the _ source frequency - spin down _ plane . results on simulated peakmaps suggest various advantages in favor of the use of the frequency hough . the ones which show up to really make the difference are 1 ) the possibility to enhance the frequency resolution without relevantly affecting the computing cost . this reduces the digitization effects ; 2 ) the excess of candidates due to local disturbances in some places of the sky map . they do not affect the new analysis because each map is constructed for only one position i

# Cross Attention

In [33]:
import torch

model.eval()

inputs = tokenizer(
    test['article'][0],
    return_tensors="pt",
    truncation=True,
    padding=True
).to("cuda")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746",attn_implementation="eager").to(device)

tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

inputs = tokenizer(
    "Your input text",
    return_tensors="pt",
    truncation=True,
    padding=True
)

# Move inputs to GPU
inputs = {k: v.to(device) for k, v in inputs.items()}

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1181.95it/s, Materializing param=model.shared.weight]                                   


In [38]:
with torch.no_grad():
    outputs = model(
        **inputs,
        output_attentions=True,
        return_dict=True
    )

In [44]:
cross_attention = outputs.cross_attentions